# Premier League Analytics — 01: Data Acquisition

Downloads and cleans two seasons of Premier League match data from football-data.co.uk.  
Run `python data/fetch_data.py` first.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

DATA = Path('data')
for f in [DATA / 'E0_2223.csv', DATA / 'E0_2324.csv']:
    if not f.exists():
        raise FileNotFoundError(f'{f} not found — run data/fetch_data.py')

## 1. Load Match Data

In [ ]:
s2223 = pd.read_csv(DATA / 'E0_2223.csv')
s2324 = pd.read_csv(DATA / 'E0_2324.csv')

s2223['season'] = '2022-23'
s2324['season'] = '2023-24'

matches = pd.concat([s2223, s2324], ignore_index=True)
print(f'Combined: {len(matches):,} matches × {matches.shape[1]} columns')
matches.head(3)

## 2. Clean & Validate

In [ ]:
# Parse date
matches['Date'] = pd.to_datetime(matches['Date'], dayfirst=True, errors='coerce')
print(f'Date nulls: {matches["Date"].isna().sum()}')

# Core numeric columns
core = ['FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']
present = [c for c in core if c in matches.columns]
for col in present:
    matches[col] = pd.to_numeric(matches[col], errors='coerce')

print(f'Null counts for core columns:')
matches[present].isna().sum()

## 3. Build Team-Level Season Summary

In [ ]:
def team_stats(df: pd.DataFrame, side: str) -> pd.DataFrame:
    """Aggregate per-team stats for home or away side."""
    prefix = 'H' if side == 'home' else 'A'
    opp    = 'A' if side == 'home' else 'H'
    team_col = 'HomeTeam' if side == 'home' else 'AwayTeam'
    return (
        df.groupby(['season', team_col]).agg(
            games=('Date', 'count'),
            goals_for=(f'FT{prefix}G', 'sum'),
            goals_against=(f'FT{opp}G', 'sum'),
            shots=(f'{prefix}S', 'sum'),
            shots_on_target=(f'{prefix}ST', 'sum'),
            corners=(f'{prefix}C', 'sum'),
            yellows=(f'{prefix}Y', 'sum'),
        )
        .rename_axis(['season', 'team'])
        .reset_index()
    )

home_stats = team_stats(matches, 'home')
away_stats = team_stats(matches, 'away')

team_sum = pd.concat([home_stats, away_stats]).groupby(['season', 'team']).sum().reset_index()
team_sum['goal_diff'] = team_sum['goals_for'] - team_sum['goals_against']
team_sum['shot_accuracy'] = team_sum['shots_on_target'] / team_sum['shots'].replace(0, np.nan)

print(f'Team summary: {len(team_sum)} rows')
team_sum.head()

## 4. Save Processed Data

In [ ]:
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
matches.to_parquet(out_dir / 'matches_clean.parquet', index=False)
team_sum.to_parquet(out_dir / 'team_season_stats.parquet', index=False)
print('Saved matches_clean.parquet and team_season_stats.parquet to outputs/')